# 01c — Dataset diagnostico completo di transizioni

Questo notebook chiude il contratto dati prima del baseline neurale. Genera transizioni da 1 ms con stato completo ai bordi, microtracce a 0,025 ms, eventi somatici e dendritici, controlli vicini alla soglia e branching da uno stato comune.

I protocolli positivi sono esattamente quelli confermati in `01b`; non vengono modificati pesi, timing, selezione del ramo, morfologia o meccanismi. Non viene ancora addestrato alcun modello.

## 1. Checkout riproducibile

In [ ]:
import importlib
import os
import subprocess
import sys
from pathlib import Path

ELM_REPOSITORY = "https://github.com/Zagred47/giada.git"
ELM_REF = os.environ.get("HAYFLOW_ELM_REF", "main")
if Path("/kaggle/working").is_dir():
    NOTEBOOK_ROOT = Path("/kaggle/working")
elif Path("/content").is_dir():
    NOTEBOOK_ROOT = Path("/content")
else:
    NOTEBOOK_ROOT = Path.cwd().resolve()
WORKSPACE = NOTEBOOK_ROOT / "hayflow_workspace"
WORKSPACE.mkdir(parents=True, exist_ok=True)

def run(command, cwd=None):
    print("+", " ".join(map(str, command)))
    subprocess.run(list(map(str, command)), cwd=cwd, check=True)

elm_override = os.environ.get("HAYFLOW_ELM_REPO")
mounted = [Path(elm_override).expanduser()] if elm_override else []
mounted.extend([Path.cwd(), *Path.cwd().parents])
mounted_elm = next((p.resolve() for p in mounted if (p / "src" / "hayflow_teacher").is_dir()), None)
if mounted_elm is not None:
    ELM_REPO = mounted_elm
else:
    ELM_REPO = WORKSPACE / "elmneuron"
    if not (ELM_REPO / ".git").is_dir():
        run(["git", "clone", ELM_REPOSITORY, ELM_REPO])
    run(["git", "fetch", "origin", ELM_REF], cwd=ELM_REPO)
    run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=ELM_REPO)
os.environ["HAYFLOW_ELM_REPO"] = str(ELM_REPO)
for module_name in tuple(sys.modules):
    if module_name == "src.hayflow_teacher" or module_name.startswith("src.hayflow_teacher."):
        sys.modules.pop(module_name, None)
importlib.invalidate_caches()
print("Owned repository:", ELM_REPO)

## 2. Teacher canonico e dipendenze

Il teacher resta separato e bloccato al commit verificato. I MOD originali vengono compilati senza modifiche.

In [ ]:
import shutil

TEACHER_REPOSITORY = "https://github.com/SelfishGene/neuron_as_deep_net.git"
TEACHER_COMMIT = "074c4666300a8ad246601dab179a97a6942f0f29"
teacher_override = os.environ.get("HAYFLOW_TEACHER_REPO")
sibling = ELM_REPO.parent / "neuron_as_deep_net"
TEACHER_REPO = Path(teacher_override).expanduser().resolve() if teacher_override else sibling
if not (TEACHER_REPO / ".git").is_dir():
    run(["git", "clone", TEACHER_REPOSITORY, TEACHER_REPO])
run(["git", "checkout", "--detach", TEACHER_COMMIT], cwd=TEACHER_REPO)
assert subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=TEACHER_REPO, text=True).strip() == TEACHER_COMMIT
os.environ["HAYFLOW_TEACHER_REPO"] = str(TEACHER_REPO)
run([sys.executable, "-m", "pip", "install", "--quiet", "neuron==8.2.7", "numpy", "pandas", "matplotlib", "h5py", "pyarrow", "pyyaml"])
SIMULATION_DIR = TEACHER_REPO / "L5PC_NEURON_simulation"
compiled = list(SIMULATION_DIR.rglob("libnrnmech.so"))
if not compiled:
    nrnivmodl = shutil.which("nrnivmodl") or str(Path(sys.executable).parent / "nrnivmodl")
    run([nrnivmodl, "mods"], cwd=SIMULATION_DIR)
assert list(SIMULATION_DIR.rglob("libnrnmech.so")), "MOD compilation failed"
print("Teacher ready:", TEACHER_REPO)

## 3. Artefatti di calibrazione `01b`

Il notebook deve poter leggere lo ZIP scaricato da `01b` oppure la sua cartella estratta. In una nuova sessione Kaggle, aggiungere `hayflow_dendritic_protocol_calibration.zip` come input del notebook. Il gate verifica realmente tutti gli 88 SHA-256.

In [ ]:
calibration_override = os.environ.get("HAYFLOW_CALIBRATION_SOURCE")
calibration_candidates = [Path(calibration_override).expanduser()] if calibration_override else []
calibration_candidates.extend([
    NOTEBOOK_ROOT / "hayflow_dendritic_protocol_calibration",
    NOTEBOOK_ROOT / "hayflow_dendritic_protocol_calibration.zip",
])
if Path("/kaggle/input").is_dir():
    calibration_candidates.extend(Path("/kaggle/input").rglob("hayflow_dendritic_protocol_calibration.zip"))
CALIBRATION_SOURCE = next((path.resolve() for path in calibration_candidates if path.exists()), None)
assert CALIBRATION_SOURCE is not None, (
    "Calibrazione 01b non trovata. Aggiungi hayflow_dendritic_protocol_calibration.zip "
    "agli input Kaggle oppure imposta HAYFLOW_CALIBRATION_SOURCE."
)
print("Calibration source:", CALIBRATION_SOURCE)

## 4. Contratto v1 e teacher manifest

Prepariamo i 642 segmenti, lo schema core da 17.220 variabili, i 9.182 target privilegiati e la sonda del centro effettivo del cluster tuft.

In [ ]:
import json
import yaml
from IPython.display import display

sys.path.insert(0, str(ELM_REPO))
from src.hayflow_data import BurnInCriteria
from src.hayflow_teacher import DiagnosticDatasetV1Session, expected_audit_hashes

BASE_CONFIG_PATH = ELM_REPO / "configs" / "hayflow" / "transition_dataset_diagnostic.yml"
V1_CONFIG_PATH = ELM_REPO / "configs" / "hayflow" / "diagnostic_dataset_v1.yml"
base_config = yaml.safe_load(BASE_CONFIG_PATH.read_text(encoding="utf-8"))
v1_config = yaml.safe_load(V1_CONFIG_PATH.read_text(encoding="utf-8"))
OUTPUT_DIR = NOTEBOOK_ROOT / "artifacts" / "diagnostic_dataset_v1"
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

session = DiagnosticDatasetV1Session(
    ELM_REPO,
    TEACHER_REPO,
    calibration_source=CALIBRATION_SOURCE,
    dataset_config_path=V1_CONFIG_PATH,
    output_dir=OUTPUT_DIR,
    seed=base_config["runtime"]["seed"],
    expected_teacher_hashes=expected_audit_hashes(),
    native_snapshot_stride=v1_config["storage"]["native_snapshot_stride_ms"],
)
teacher_report = session.prepare_teacher()
contract_report = session.prepare_v1_contract()
display({"teacher": teacher_report, "v1_contract": contract_report})
assert teacher_report["segment_count"] == 642
assert contract_report["core_state_width"] == 17220
assert contract_report["privileged_state_width"] == 9182

## 5. Burn-in e corrente somatica

Il riposo viene misurato, non scelto arbitrariamente. Verifichiamo anche che il driver somatico sia deterministico. La corrente paired resta quella confermata dal protocollo Ca; una seconda calibrazione misura separatamente il vero protocollo a impulso singolo, senza estrapolarne la soglia.

In [ ]:
burnin_config = dict(base_config["burnin"])
burnin_config["slow_mechanisms"] = tuple(burnin_config["slow_mechanisms"])
burnin_report = session.run_burn_in(BurnInCriteria(**burnin_config))
current_smoke = session.run_somatic_current_smoke_test(**base_config["somatic_current"]["smoke_test"])
paired_somatic_calibration = session.calibrate_somatic_spike_current(
    candidate_amplitudes_na=base_config["somatic_current"]["calibration"]["candidate_amplitudes_na"]
)
single_somatic_calibration = session.calibrate_somatic_single_spike_current(
    candidate_amplitudes_na=base_config["somatic_current"]["calibration"]["candidate_amplitudes_na"]
)
display({
    "burnin_ms": burnin_report["burnin_duration_ms"],
    "current_smoke": current_smoke,
    "paired_somatic_calibration": paired_somatic_calibration,
    "single_somatic_calibration": single_somatic_calibration,
})
assert burnin_report["converged"] and current_smoke["valid"]
assert paired_somatic_calibration["valid"] and single_somatic_calibration["valid"]

## 6. Piano delle traiettorie

Gli split sono assegnati a intere traiettorie. I positivi dendritici usano i tre seed confermati; i controlli modificano numero di sinapsi, pairing o finestra temporale senza toccare le ricette positive. I cinque futuri di branching condividono stato iniziale e chiave Random123.

In [ ]:
import pandas as pd

dendritic_duration = v1_config["confirmed_protocols"]["nmda_plateau"]["duration_ms"]
protocols = session.build_v1_protocols(dendritic_duration_ms=dendritic_duration)
protocol_table = pd.DataFrame(session.protocol_rows)
display(protocol_table[[
    "trajectory_id", "category", "protocol_id", "protocol_variant",
    "seed", "duration_ms", "split", "negative_control"
]])
print("Traiettorie:", len(protocols))
print("Transizioni previste:", sum(item.duration_ms for item in protocols))
print("Snapshot nativi previsti (circa):", sum((item.duration_ms + session.native_snapshot_stride - 1) // session.native_snapshot_stride for item in protocols))

## 7. Preflight bloccante

Prima di scrivere l'HDF5 eseguiamo soltanto pochi millisecondi: i sei protocolli confermati vengono riprodotti sia dal percorso di calibrazione sia dal percorso di storage corrente e devono coincidere esattamente tra loro. Le vecchie tracce `01b`, prodotte prima della correzione post-restore di `fcurrent()`, restano provenienza con hash verificato e il loro drift viene documentato senza usarlo come falso gate. Inoltre il protocollo single-pulse deve produrre spike soma/AIS e i cinque branching devono condividere stato completo e RNG. Se uno solo dei controlli correnti fallisce, il notebook si ferma qui.

In [ ]:
preflight_report = session.run_v1_preflight(
    protocols,
    prefix_duration_ms=v1_config["preflight"]["confirmed_prefix_duration_ms"],
)
display({
    "valid": preflight_report["valid"],
    "protocol_plan_sha256": preflight_report["protocol_plan_sha256"],
    "corrected_runtime_prefixes_valid": preflight_report["confirmed_prefixes"]["valid"],
    "corrected_runtime_max_error": preflight_report["confirmed_prefixes"]["maximum_error"],
    "historical_01b_drift_max_error_non_gating": preflight_report["historical_01b_trace_drift"]["maximum_error"],
    "single_spike": preflight_report["single_spike"],
    "branching_initial_state": preflight_report["branching_initial_state"],
})
assert preflight_report["valid"], preflight_report
print("Preflight superato: la generazione completa puo iniziare.")

## 8. Generazione del dataset

Ogni bordo contiene stato core, target privilegiati e RNG. Le microtracce includono tutti i 642 voltaggi, le sonde canoniche, il centro del cluster e osservabili Ca/AMPA/NMDA. Un checkpoint nativo ogni 10 ms consente di riprodurre ogni transizione tramite un breve replay del prefisso. Durante la generazione viene mostrato un tracker live con transizioni completate, percentuale, velocità, tempo trascorso, ETA, traiettoria e step correnti.

In [ ]:
dataset_manifest = session.generate_dataset(protocols)
display({
    "schema_version": dataset_manifest["schema_version"],
    "trajectory_count": dataset_manifest["trajectory_count"],
    "transition_count": dataset_manifest["transition_count"],
    "event_counts": dataset_manifest["event_counts"],
    "size_estimate": dataset_manifest["size_estimate"],
})
print("Transition store:", session.transition_path)

## 9. Validazione completa

Il gate riproduce sequenzialmente tutte le transizioni, controlla eventi e censura, split, branching, schema, NaN/Inf, timestamp, 88 hash originali e overlap esatto dei primi 35 ms. Le quattro fasi della validazione e il replay esaustivo espongono avanzamento ed ETA. In caso di errore il report viene comunque lasciato nell'archivio per la diagnosi.

In [ ]:
validation_error = None
try:
    validation_report = session.validate_dataset_v1(replay_count=5)
except RuntimeError as error:
    validation_error = error
    validation_report = json.loads((OUTPUT_DIR / "validation_report.json").read_text(encoding="utf-8"))
display(validation_report)
if validation_error is not None:
    print("VALIDATION BLOCKED:", validation_error)
else:
    print("Dataset diagnostico v1 validato.")

## 10. Figure diagnostiche

Le figure usano tempo relativo allo stimolo e pannelli separati per voltaggio, calcio, corrente di calcio, conduttanza NMDA e corrente NMDA. Le scale sono condivise tra i tre seed dello stesso protocollo.

In [ ]:
from IPython.display import Image, display

for figure_path in sorted(session.figures_dir.glob("confirmed_*.png")):
    print(figure_path.name)
    display(Image(filename=str(figure_path)))

## 11. Controllo degli output

In [ ]:
required_outputs = [
    "dataset_manifest.json", "state_schema.json",
    "transition_index.parquet", "protocols.parquet", "splits.json",
    "events.parquet", "branching_index.parquet",
    "somatic_current_calibration.json",
    "somatic_single_spike_current_calibration.json",
    "storage_report.json", "preflight_report.json", "validation_report.json",
    "prefix_overlap_report.json", "transition_dataset.h5",
    "artifact_index.json",
]
missing_outputs = [name for name in required_outputs if not (OUTPUT_DIR / name).is_file()]
assert not missing_outputs, f"Output mancanti: {missing_outputs}"
print("Output completi:", len(required_outputs))

## 12. ZIP e download diretto

Questa è la procedura Base64 + Blob usata stabilmente nel progetto per scaricare da Kaggle.

In [ ]:
from base64 import b64encode
from pathlib import Path
from shutil import make_archive
from IPython.display import Javascript, display

artifact_dir = Path(session.output_dir).resolve()
zip_base = Path("/kaggle/working/hayflow_diagnostic_dataset_v1")
zip_path = Path(make_archive(str(zip_base), "zip", root_dir=artifact_dir.parent, base_dir=artifact_dir.name))
print(f"ZIP creato: {zip_path}")
print(f"Dimensione: {zip_path.stat().st_size / 1024**2:.2f} MB")
encoded = b64encode(zip_path.read_bytes()).decode("ascii")
display(Javascript(f"""
(() => {{
    const binary = atob("{encoded}");
    const bytes = new Uint8Array(binary.length);
    for (let i = 0; i < binary.length; i++) {{
        bytes[i] = binary.charCodeAt(i);
    }}
    const blob = new Blob([bytes], {{type: "application/zip"}});
    const url = URL.createObjectURL(blob);
    const link = document.createElement("a");
    link.href = url;
    link.download = "hayflow_diagnostic_dataset_v1.zip";
    document.body.appendChild(link);
    link.click();
    link.remove();
    setTimeout(() => URL.revokeObjectURL(url), 10000);
}})();
"""))

## 13. Gate finale

Il notebook è concluso soltanto con `valid: true`. Se il gate fallisce, conservare lo ZIP e analizzare `validation_report.json`; non modificare soglie o ricette positive per forzare il risultato.

In [ ]:
assert validation_report["valid"], validation_report["blockers"]
assert validation_report["exhaustive_transition_replay"]["valid"]
assert validation_report["contract"]["prefix_overlap"]["maximum_error"] == 0.0
assert validation_report["contract"]["required_events_not_right_censored"]
print("01c completato: il contratto dati è pronto per 02_full_state_flowmap_baseline.ipynb.")